## Customer Order Data Display

In [ ]:
%%capture cap --no-display

%load_ext autoreload
%autoreload 2

# --- IMPORTS ---
import importlib, html
from datetime import datetime
from collections import defaultdict

import app as app_mod; importlib.reload(app_mod)
from app import create_app, db

import models as models_mod; importlib.reload(models_mod)
from models import Customer


# --- CONFIG ---
# Set this based on your Customer model:
#   True  => model has first_name + last_name
#   False => model has full_name
HAS_SEPARATE_NAMES = True  # change to False if you use full_name

# Optional: If you have a 'product' column in Customer, set True for nicer tables
HAS_PRODUCT_COL = "product" in Customer.__table__.columns.keys()
print("HAS_PRODUCT_COL (from model):", HAS_PRODUCT_COL)


# --- QUERY ---
app = create_app()
with app.app_context():
    customers = Customer.query.order_by(Customer.created_at.asc()).all()

# Helper to get display name depending on schema
def get_name(c):
    if HAS_SEPARATE_NAMES:
        return f"{getattr(c,'first_name','').strip()} {getattr(c,'last_name','').strip()}".strip()
    return getattr(c, "full_name", "").strip()

# --- BUILD STRUCTURES ---
# 1) Customer list (all names, including duplicates)
customers_list = [get_name(c) for c in customers]

# 2) Customer set (unique names)
customers_set = set(customers_list)

# 3) Customer dictionary (name -> product or -> category if no product column)

customers_dict_list = defaultdict(list)

if HAS_PRODUCT_COL:
    for c in customers:
        name = get_name(c)
        product = c.product
        if product and product not in customers_dict_list[name]:
            customers_dict_list[name].append(product)
else:
    for c in customers:
        name = get_name(c)
        category = c.category
        if category and category not in customers_dict_list[name]:
            customers_dict_list[name].append(category)

customers_dict = {name: ", ".join(values) for name, values in customers_dict_list.items()}


# 4) Customer tuples (detailed)
#    Adjust columns to match your model; keeping product optional.
def row_tuple(c):
    name = get_name(c)
    base = (name, getattr(c, "category", None), c.price)
    if HAS_PRODUCT_COL:
        # (Name, Product, Category, Email, Phone, Price)
        return (name,  getattr(c,"product",None), *base[1:])
    # (Name, Category, Email, Phone, Price)
    return base

customer_details_tuples = [row_tuple(c) for c in customers]

# Figure out tuple headers (aligned with row_tuple)
if HAS_PRODUCT_COL:
    tuple_headers = ["Customer Name", "Product", "Category", "Price"]
else:
    tuple_headers = ["Customer Name", "Category", "Price"]

# --- RENDER HELPERS ---
def html_escape(s):
    return html.escape("" if s is None else str(s))

def render_list_as_ul(items):
    return "<ul>\n" + "\n".join(f"  <li>{html_escape(x)}</li>" for x in items) + "\n</ul>"

def render_set_as_ul(items_set):
    # sort for stable output
    items = sorted(items_set)
    return render_list_as_ul(items)

def render_dict_as_table(d):
    rows = "".join(
        f"<tr><td>{html_escape(k)}</td><td>{html_escape(v)}</td></tr>"
        for k, v in d.items()
    )
    return f"""
<table class="kv">
  <thead><tr><th>Customer Name</th><th>{'Product' if HAS_PRODUCT_COL else 'Category'}</th></tr></thead>
  <tbody>
    {rows}
  </tbody>
</table>
"""

def render_tuples_as_table(headers, rows):
    thead = "".join(f"<th>{html_escape(h)}</th>" for h in headers)
    body_rows = []
    for r in rows:
        tds = "".join(f"<td>{html_escape(x)}</td>" for x in r)
        body_rows.append(f"<tr>{tds}</tr>")
    return f"""
<table class="grid">
  <thead><tr>{thead}</tr></thead>
  <tbody>
    {"".join(body_rows)}
  </tbody>
</table>
"""

# --- PAGE STYLES ---
CSS = """
<style>
  body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 24px; color: #222; }
  h1 { margin: 0 0 4px 0; }
  h2 { margin: 24px 0 8px 0; }
  .muted { color: #666; }
  .section { margin-bottom: 28px; }
  .scroll { max-height: 420px; overflow: auto; border: 1px solid #e5e7eb; padding: 12px; border-radius: 8px; background: #fafafa; }
  table { border-collapse: collapse; width: 100%; }
  table.grid th, table.grid td, table.kv th, table.kv td { border: 1px solid #e5e7eb; padding: 8px; }
  table.grid th { background: #f3f4f6; text-align: left; }
  table.kv th { background: #f9fafb; text-align: left; }
  code.kv { background: #f6f8fa; padding: 2px 6px; border-radius: 4px; }
  .pill { display: inline-block; padding: 2px 8px; background: #eef2ff; color: #3730a3; border-radius: 999px; font-size: 12px; }
  .row { display: grid; grid-template-columns: 1fr; gap: 16px; }
  @media (min-width: 900px) { .row { grid-template-columns: 1fr 1fr; } }
</style>
"""

# --- COMPOSE HTML ---
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

html_report = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<title>Customer Insights Report</title>
{CSS}
</head>
<body>
  <h1>Customer Insights Report</h1>
  <div class="muted">Generated at: {html_escape(generated_at)}</div>

  <div class="section">
    <h2>1) Customer Tuples <span class="pill">Detailed rows</span></h2>
    <div class="scroll">
      {render_tuples_as_table(tuple_headers, customer_details_tuples)}
    </div>
    <div class="muted">Showing {len(customer_details_tuples)} rows.</div>
  </div>

  <div class="row">
    <div class="section">
      <h2>2) Customer List <span class="pill">Duplicates allowed</span></h2>
      <div class="scroll">
        {render_list_as_ul(customers_list)}
      </div>
      <div class="muted">List length: <code class="kv">{len(customers_list)}</code></div>
    </div>

    <div class="section">
      <h2>3) Customer Set <span class="pill">Unique names</span></h2>
      <div class="scroll">
        {render_set_as_ul(customers_set)}
      </div>
      <div class="muted">Unique names: <code class="kv">{len(customers_set)}</code></div>
    </div>
  </div>

  <div class="section">
    <h2>4) Customer Dictionary <span class="pill">Name → {'Product' if HAS_PRODUCT_COL else 'Category'}</span></h2>
    <div class="scroll">
      {render_dict_as_table(customers_dict)}
    </div>
    <div class="muted">Pairs: <code class="kv">{len(customers_dict)}</code></div>
  </div>

 <div class="section">
    <h2>Raw Data From Customers Database</h2>
    <div class="scroll"><pre class="prewrap">{captured_stdout_html}</pre></div>
  </div>


</body>
</html>
"""

# --- WRITE FILE ---
out_path = "customer_report.html"
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html_report)

print(f"Wrote: {out_path}")

In [ ]:
import webbrowser, os
webbrowser.open('file://' + os.path.abspath('customer_report.html'))